In [2]:
import time
from sage.all import *

# 参数保持不变
p = 434252269029337012720086440207
a = -35
b = 98
F = GF(p)
E = EllipticCurve(F, [a, b])

G = E(16378704336066569231287640165, 377857010369614774097663166640)
A = E(0x54b35f407996eb8fc566d3b12, 0xc857b1b70d5ef09507bf8bd6)

n = G.order()
print("G.order() =", n)
print("n 是否光滑:", factor(n))

# --- 如果你想演示 MOV 攻击 ---
k = 2
# 修正：modulus 应该使用多项式环变量
R_poly.<x> = PolynomialRing(F)
K.<w> = GF(p^k, modulus=x^2+1)
E_ext = E.change_ring(K)
G_ext = E_ext(G)
A_ext = E_ext(A)

print("正在寻找辅助点 T (已优化跳过 E_ext.order())...")
# 技巧：不需要计算完整的 order_ext，利用 (p+1) 的倍数构造即可
# 对于大部分 k=2 的超奇异或特定曲线，点阶性质比较特殊
while True:
    T = E_ext.random_point()
    # 尝试消去非目标子群分量，确保 T 在 n-torsion 附近
    # 如果不知道具体倍数，可以直接尝试原始随机点，Weil 配对依然有效
    alpha = G_ext.weil_pairing(T, n)
    if alpha != 1:
        break

beta = A_ext.weil_pairing(T, n)

print("开始计算扩域离散对数...")
start = time.time()
# 显式指定 ord=n 触发 Pohlig-Hellman 加速
na = beta.log(alpha) 
elapsed = time.time() - start

print(f"解得私钥 na = {na}")
print(f"耗时 = {elapsed:.4f} 秒")

# 修正断言：A = na * G
assert na * G == A
print("✅ Success: 验证通过！")

# --- 额外建议 ---
# print("对比实验：直接对原始曲线求对数...")
# start2 = time.time()
# na_direct = G.discrete_log(A)
# print(f"直接破解耗时: {time.time() - start2:.4f} 秒")

G.order() = 434252269029337012720086440208
n 是否光滑: 2^4 * 3 * 73 * 88591 * 3882601 * 360301137196997
正在寻找辅助点 T (已优化跳过 E_ext.order())...
开始计算扩域离散对数...
解得私钥 na = 6643966786389654634
耗时 = 83.7683 秒
✅ Success: 验证通过！
